# Data Cleaning

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ta
from pathlib import Path

In [16]:
PROJECT_ROOT = Path('__file__').resolve().parent.parent
csv_path = PROJECT_ROOT / "data" / "raw_stock_data.csv" # setting the csv raw file path
csv_path

WindowsPath('D:/IndianStockMarketAnalysis/data/raw_stock_data.csv')

In [29]:
df = pd.read_csv(csv_path)
df.head()

,Date,Close,High,Low,Open,Volume,Stock
0,2025-03-20,872.580322,875.170313,862.985046,865.303681,17833978,HDFC BANK
1,2025-03-21,873.369690,875.219684,866.068418,866.068418,33508264,HDFC BANK
2,2025-03-24,887.996948,890.167619,874.997672,877.636984,17393736,HDFC BANK
3,2025-03-25,898.578918,909.555548,888.736976,890.414316,39101416,HDFC BANK
4,2025-03-26,891.228333,901.563607,888.983649,900.330278,24478442,HDFC BANK


In [33]:
df["Close"].dtype

dtype('float64')

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1245 entries, 0 to 1244
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    1245 non-null   object 
 1   Close   1245 non-null   float64
 2   High    1245 non-null   float64
 3   Low     1245 non-null   float64
 4   Open    1245 non-null   float64
 5   Volume  1245 non-null   int64  
 6   Stock   1245 non-null   object 
dtypes: float64(4), int64(1), object(2)
memory usage: 68.2+ KB


In [34]:
# convert Date to DATETIME
df["Date"] = pd.to_datetime(df["Date"])

# numerical cols
numeric_cols = ["Open","High","Low","Close","Volume"]

# cat cols
cat_cols = ["Stock"]


In [35]:
# sort the data 
df = df.sort_values(["Stock","Date"])
df = df.reset_index(drop=True)

In [ ]:
df.duplicated().sum() # check duplicates

np.int64(0)

In [ ]:
df.isnull().sum() # check Missing Values

Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
Stock     0
dtype: int64

In [37]:
df.head()

,Date,Close,High,Low,Open,Volume,Stock
0,2025-03-20,872.580322,875.170313,862.985046,865.303681,17833978,HDFC BANK
1,2025-03-21,873.369690,875.219684,866.068418,866.068418,33508264,HDFC BANK
2,2025-03-24,887.996948,890.167619,874.997672,877.636984,17393736,HDFC BANK
3,2025-03-25,898.578918,909.555548,888.736976,890.414316,39101416,HDFC BANK
4,2025-03-26,891.228333,901.563607,888.983649,900.330278,24478442,HDFC BANK


# Feature Engineering

In [49]:
df["Return"] = df.groupby("Stock")["Close"].pct_change()

In [50]:
# moving average
df["MA20"] = df.groupby("Stock")["Close"].transform(lambda x: x.rolling(20).mean())

df["MA50"] = df.groupby("Stock")["Close"].transform(lambda x: x.rolling(50).mean())


In [61]:
df[df['Stock']=="TCS"].head(30)

,Date,Close,High,Low,Open,Volume,Stock,Return,MA20,MA50,Volatility
996,2025-03-20,3444.123779,3459.396031,3394.682527,3394.682527,2845192,TCS,NaN,NaN,NaN,NaN
997,2025-03-21,3458.574707,3495.450223,3349.252685,3358.918638,4161925,TCS,0.004196,NaN,NaN,NaN
998,2025-03-24,3507.726074,3527.831304,3447.458997,3479.743187,1834751,TCS,0.014211,NaN,NaN,NaN
999,2025-03-25,3535.563965,3586.068569,3516.473708,3523.239875,3135390,TCS,0.007936,NaN,NaN,NaN
1000,2025-03-26,3514.347168,3557.553930,3504.729592,3525.897934,1734499,TCS,-0.006001,NaN,NaN,NaN
1001,2025-03-27,3529.232666,3540.638538,3489.892285,3494.097069,2528474,TCS,0.004236,NaN,NaN,NaN
1002,2025-03-28,3485.687256,3538.656721,3471.913274,3529.039146,2051919,TCS,-0.012338,NaN,NaN,NaN
1003,2025-04-01,3432.186523,3464.277439,3407.248318,3453.354865,2618493,TCS,-0.015349,NaN,NaN,NaN
1004,2025-04-02,3425.565430,3445.042467,3412.129849,3425.613807,1764313,TCS,-0.001929,NaN,NaN,NaN
1005,2025-04-03,3289.468750,3385.016791,3282.702583,3374.384243,4537821,TCS,-0.039730,NaN,NaN,NaN


In [58]:
df["Volatility"] = df.groupby("Stock")["Return"].transform(lambda x: x.rolling(20).std())

In [62]:
df["RSI"] = df.groupby("Stock")["Close"].transform(
    lambda x: ta.momentum.RSIIndicator(x).rsi()
)

In [63]:
def macd_calc(x):
    macd = ta.trend.MACD(x)
    return macd.macd()

df["MACD"] = df.groupby("Stock")["Close"].transform(macd_calc)

In [64]:
conditions = [
    (df["Close"] > df["MA50"]) & (df["RSI"] > 55),
    (df["Close"] < df["MA50"])
]

choices = ["Bullish","Bearish"]

df["Trend"] = np.select(conditions, choices, default="Sideways")

In [66]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1245 entries, 0 to 1244
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        1245 non-null   datetime64[ns]
 1   Close       1245 non-null   float64       
 2   High        1245 non-null   float64       
 3   Low         1245 non-null   float64       
 4   Open        1245 non-null   float64       
 5   Volume      1245 non-null   int64         
 6   Stock       1245 non-null   object        
 7   Return      1240 non-null   float64       
 8   MA20        1150 non-null   float64       
 9   MA50        1000 non-null   float64       
 10  Volatility  1145 non-null   float64       
 11  RSI         1180 non-null   float64       
 12  MACD        1120 non-null   float64       
 13  Trend       1245 non-null   object        
dtypes: datetime64[ns](1), float64(10), int64(1), object(2)
memory usage: 136.3+ KB


In [67]:
df.tail()

,Date,Close,High,Low,Open,Volume,Stock,Return,MA20,MA50,Volatility,RSI,MACD,Trend
1240,2026-03-16,2409.199951,2425.000000,2366.100098,2410.0,3807907,TCS,-0.000539,2587.629980,2892.890068,0.011765,18.942903,-130.754108,Bearish
1241,2026-03-17,2391.699951,2420.000000,2360.000000,2420.0,2993367,TCS,-0.007264,2571.884973,2876.870850,0.011489,18.155409,-131.379193,Bearish
1242,2026-03-18,2440.800049,2482.899902,2410.000000,2410.0,3082388,TCS,0.020529,2558.054980,2862.513276,0.012777,27.288777,-126.454916,Bearish
1243,2026-03-19,2356.000000,2424.000000,2350.199951,2417.0,2757060,TCS,-0.034743,2541.109985,2845.679878,0.014375,22.598316,-127.920459,Bearish
1244,2026-03-20,2390.600098,2407.800049,2364.199951,2386.0,4205705,TCS,0.014686,2526.744995,2828.756694,0.015143,28.033657,-124.850772,Bearish


In [69]:
save_path = PROJECT_ROOT / "data" / "processed_stock_data.csv"
df.to_csv(save_path, index=False)